# 08 · Contenedores: API e interfaz con Docker Compose

**Propósito.** Documentar cómo empaquetamos la API y la interfaz de TubeCast en dos imágenes separadas, cómo las conectamos con red host y cómo comprobamos el montaje completo en la computadora local antes de llevarlo a EC2 ([09-despliegue](09-despliegue.ipynb)).

Este notebook **no arranca servidores ni construye imágenes**: revisa los archivos, comprueba que la configuración es coherente y documenta lo que observamos al ejecutar los comandos en la terminal. Las capturas están en `evidencia/`.

| Elemento | API | Interfaz |
|---|---|---|
| Servicio de Compose | `api` | `ui` |
| Imagen y tag | `tubecast-api-image:v1` | `tubecast-ui-image:v1` |
| Contenedor | `tubecast-api-container` | `tubecast-ui-container` |
| Dockerfile | `src/api/Dockerfile` | `src/frontend/Dockerfile` |
| Contexto de construcción | raíz del repositorio | raíz del repositorio |
| Proceso | `uvicorn src.api.main:app` | `streamlit run src/frontend/app.py` |
| Escucha | `127.0.0.1:8000` (`API_HOST`) | `0.0.0.0:8501` |
| Configuración | `.env` de la raíz (`env_file`) | sólo `API_URL=http://127.0.0.1:8000` |

`v1` es el tag de la imagen, no la versión del modelo. La versión del modelo la decide el alias `champion` al arrancar la API.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Los notebooks se ejecutan desde la raíz del repositorio para compartir rutas con src/
RAIZ = Path.cwd().parent if Path.cwd().name in {"notebooks", "informe"} else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("Raíz del repositorio:", RAIZ)

## 1. Dependencias: una sola fuente, el lock

Las dos imágenes usan Python 3.12 (`python:3.12-slim`) e instalan `requirements.txt`, que **no se edita a mano**: se genera desde `uv.lock`.

```bash
uv export --locked --no-emit-project --no-hashes --no-dev --output-file requirements.txt
```

La celda siguiente vuelve a exportar el lock a un archivo temporal y lo compara con el `requirements.txt` versionado. Si alguien cambia dependencias y olvida regenerarlo, aquí falla.

In [ ]:
import tempfile, difflib

with tempfile.TemporaryDirectory() as tmp:
    destino = Path(tmp) / "req.txt"
    subprocess.run(
        ["uv", "export", "--locked", "--no-emit-project", "--no-hashes", "--no-dev",
         "--output-file", str(destino)],
        check=True, capture_output=True, text=True,
    )
    exportado = [l for l in destino.read_text().splitlines() if not l.startswith("#")]

versionado = [l for l in Path("requirements.txt").read_text().splitlines() if not l.startswith("#")]
diferencias = list(difflib.unified_diff(versionado, exportado, lineterm=""))
paquetes = [l for l in versionado if l and not l.startswith(" ")]
print(f"requirements.txt: {len(paquetes)} paquetes fijados")
print("Coincide con uv.lock:", not diferencias)
assert not diferencias, "Regenera requirements.txt con uv export"
for clave in ("fastapi", "mlflow", "scikit-learn", "skops", "streamlit", "uvicorn"):
    print(" ", next(l for l in paquetes if l.startswith(clave + "==")))

**Interpretación.** Las imágenes instalan exactamente las mismas versiones con las que entrenamos (`scikit-learn`, `mlflow`, `skops`), que es lo que permite que el pipeline registrado se deserialice igual dentro del contenedor. Instalar el mismo archivo en las dos imágenes hace la de la interfaz más pesada de lo necesario; lo aceptamos para no mantener una segunda lista a mano.

## 2. Dockerfiles

Ambos copian primero `requirements.txt` e instalan dependencias (esa capa se reutiliza mientras el lock no cambie) y después **sólo** el código de su componente. Ningún Dockerfile copia `.env`, datos ni modelos.

In [ ]:
for ruta in ("src/api/Dockerfile", "src/frontend/Dockerfile"):
    print(f"==== {ruta}")
    print(Path(ruta).read_text())

| Instrucción | Qué hace aquí |
|---|---|
| `FROM python:3.12-slim` | Base Linux con Python 3.12 |
| `WORKDIR /code` | Directorio interno; `src/` queda en `/code/src` y se importa como paquete |
| `COPY requirements.txt` + `RUN pip install` | Dependencias bloqueadas |
| `COPY src/api` / `COPY src/frontend` | Sólo el código del componente |
| `EXPOSE` | Documenta el puerto (con red host no publica nada) |
| `CMD` | Arranque predeterminado; Compose lo reemplaza con `command` para dejarlo explícito |

**Contexto de construcción.** Usamos la raíz del repositorio porque la API es el paquete `src.api` y `requirements.txt` está en la raíz. Al ser un contexto grande, `.dockerignore` es lo que evita mandar al build lo que no debe entrar.

In [ ]:
ignorados = [l.strip() for l in Path(".dockerignore").read_text().splitlines() if l.strip() and not l.startswith("#")]
print(".dockerignore excluye:", ", ".join(ignorados))
for obligatorio in (".env", ".venv/", ".git/", "data/", "artifacts/", "*.pem"):
    assert obligatorio in ignorados, f"Falta {obligatorio} en .dockerignore"

# .env tampoco se versiona; .env.example sí, con valores ficticios
r = subprocess.run(["git", "check-ignore", ".env"], capture_output=True, text=True)
print("git ignora .env:", r.stdout.strip() == ".env")
plantilla = Path(".env.example").read_text()
assert "dapi" not in plantilla and "AIza" not in plantilla, "La plantilla parece contener un secreto real"
print("Variables de .env.example:", [l.split("=")[0] for l in plantilla.splitlines() if "=" in l and not l.startswith("#")])

## 3. Compose con red host

`docker-compose.yaml` en la raíz define los dos servicios. Ambos usan `network_mode: host`.

In [ ]:
import yaml

compose = yaml.safe_load(Path("docker-compose.yaml").read_text())
print("Proyecto Compose:", compose["name"])
for nombre, svc in compose["services"].items():
    print(f"\n[{nombre}]")
    for clave in ("image", "container_name", "build", "network_mode", "env_file", "environment", "volumes", "command", "depends_on"):
        if clave in svc:
            print(f"  {clave}: {svc[clave]}")
    assert svc["network_mode"] == "host" and "ports" not in svc

assert "env_file" not in compose["services"]["ui"], "La interfaz no debe recibir el .env con credenciales"
assert compose["services"]["ui"]["environment"]["API_URL"] == "http://127.0.0.1:8000"

### ¿Qué significa `localhost` aquí?

Sin red host, cada contenedor tiene su propia red y `127.0.0.1` dentro del contenedor de la interfaz sería **la interfaz misma**: Streamlit no encontraría la API aunque los dos puertos estuvieran publicados (lo vimos en la clase 19).

Con `network_mode: host` los dos procesos siguen aislados en archivos y dependencias, pero comparten la red del anfitrión. Entonces `127.0.0.1:8000` es el puerto 8000 **del anfitrión** (mi computadora o la instancia EC2), donde escucha la API. Por eso:

- la interfaz usa la misma URL que cuando corríamos todo sin Docker;
- no hay `ports` en el YAML (con host Docker los ignoraría);
- la API puede escuchar sólo en `127.0.0.1` y aun así la interfaz la alcanza, mientras que desde fuera del servidor no es accesible.

`API_HOST` se lee del `.env` de la raíz al interpolar el YAML. En EC2 vale `127.0.0.1`; en Docker Desktop lo podemos poner en `0.0.0.0` sólo para abrir `/docs` desde el navegador.

**Healthcheck.** El servicio `api` consulta `/health` y sólo se marca `healthy` cuando `model_loaded` es verdadero. Así `docker compose ps` dice directamente si la API ya cargó el modelo.

In [ ]:
# Validación del YAML sin imprimir valores del .env (requiere Docker instalado y un .env)
if Path(".env").exists():
    r = subprocess.run(["docker", "compose", "-f", "docker-compose.yaml", "config", "--quiet"],
                       capture_output=True, text=True)
    print("docker compose config --quiet ->", "OK" if r.returncode == 0 else r.stderr)
else:
    print("No hay .env en esta copia; se omite la validación con Docker.")

In [ ]:
# Mismo procesamiento en entrenamiento e inferencia: la API arma las mismas columnas
from src.entrenamiento.datos import LOOKBACK as LOOKBACK_ENTRENAMIENTO
from src.api import main as api_main

columnas_entrenamiento = [f"lag_{i}" for i in range(LOOKBACK_ENTRENAMIENTO, 0, -1)]
assert api_main.FEATURE_NAMES == columnas_entrenamiento
print("La API construye las mismas", len(api_main.FEATURE_NAMES), "columnas que el entrenamiento:",
      api_main.FEATURE_NAMES[0], "...", api_main.FEATURE_NAMES[-1])

## 4. Ejecución local

Requisito en Windows/macOS: Docker Desktop 4.34 o posterior con **Settings → Resources → Network → Enable host networking**.

Desde la raíz del repositorio:

```bash
uv sync --locked
cp .env.example .env        # y completar con mis accesos
docker compose -f docker-compose.yaml config --quiet
docker compose -f docker-compose.yaml build
docker compose -f docker-compose.yaml up
```

En otra terminal:

```bash
docker compose -f docker-compose.yaml ps
docker compose -f docker-compose.yaml logs api
curl http://127.0.0.1:8000/health      # con API_HOST=0.0.0.0 en Docker Desktop
```

Señales de que todo está listo:

1. En `logs api`: `Modelo cargado: registry databricks-uc: workspace.default.pcd_otono_2026_tubecast@champion -> versión N` y después `Application startup complete`.
2. `docker compose ps` muestra `tubecast-api-container` como `healthy` y `tubecast-ui-container` como `running`.
3. En `http://127.0.0.1:8501`, **Obtener Predicción** devuelve vistas estimadas y la versión `vN`.

Para terminar, sólo los contenedores de este proyecto:

```bash
docker compose -f docker-compose.yaml down
```

## 5. Comprobaciones y evidencias

**Ambiente y builds** — `uv sync --locked` y la construcción de las dos imágenes.

![Ambiente y builds](../evidencia/t8-01-uv-sync-builds.png)

*Interpretación:* `uv sync --locked` termina sin cambiar el lock y `docker compose build` construye `tubecast-api-image:v1` y `tubecast-ui-image:v1` con Python 3.12. **Resultado observado:** _(completar: tiempo aproximado del build y tamaño de las imágenes con `docker images tubecast-*`)_.

**Integración local** — estado de los servicios y una predicción desde la interfaz en contenedores.

![Integración local](../evidencia/t8-02-integracion-local.png)

*Interpretación:* los dos contenedores corren con red host, la API aparece `healthy` y la interfaz obtiene una predicción por HTTP. **Resultado observado:** _(completar: entrada usada, vistas estimadas y versión `vN` mostradas; la misma versión aparece en `logs api`)_.

## 6. Problemas que encontramos

| Síntoma | Causa | Solución |
|---|---|---|
| _(completar si aparece alguno; por ejemplo, la interfaz no conecta porque host networking no estaba activado en Docker Desktop)_ | | |

## 7. Limitaciones

- Las dos imágenes instalan todas las dependencias del lock (incluidas Prefect y Optuna, que sólo usa el entrenamiento). Pesan más de lo necesario, pero conservamos una sola fuente de versiones.
- La red host no aísla la red del contenedor respecto al anfitrión; la protección real la dan la dirección de escucha (`127.0.0.1` para la API) y el grupo de seguridad en EC2.
- La API necesita acceso de red a Databricks al arrancar. Si no lo tiene, queda en `sin_modelo` y lo dice en `/health`; sólo con `MODEL_SOURCE=local` usa un artefacto propio, y lo reporta como tal.